In [8]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
filenames = ["embedder.py", "ingest.py"]
dp = None
for filename in filenames:
    for directory in [current, *current.parents]:
        if (directory / "embed" / filename).exists():
            dp = "embed"
            sys.path.insert(0, str(directory))
            break
        if (directory / "src" / filename).exists():
            dp = "src"
            sys.path.insert(0, str(directory))
            break
    else:
        raise FileNotFoundError(f"Cannot find {dp}/{filename}")

In [4]:
from embed.embedder import Embedder

embed = Embedder(
    path="../models/Xenova/all-MiniLM-L6-v2"
)

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [5]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [6]:
v2.dot(dv)

np.float64(0.019730422395141473)

In [9]:
from src.ingest import load_faq_data

documents = load_faq_data()

In [10]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [11]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/28 [00:00<?, ?it/s]

In [12]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}